# Chapter 7 — Orthogonality and Least Squares: SageMath Companion

Symbolic, exact-arithmetic counterpart to `code/python/07_orthogonality.ipynb`. We use:

- `Matrix(QQ, ...)` for exact rational matrices (dot products, normal equations, projections).
- `Matrix(AA, ...)` for **algebraic real** matrices when square roots appear (orthonormal bases, `Q` from QR).
- `PolynomialRing(QQ)` with manual `(p*q).integral(x)` evaluation for the **L² inner product** on functions.

Everything below is exact — no floating-point error. The Legendre polynomials come out as exact rational polynomials.

**To run this notebook:** open in [CoCalc](https://cocalc.com/) (no install needed), or locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. ONB coordinates by dot products
2. Projection onto a line and onto a plane in ℝ³
3. Gram–Schmidt symbolically — `M.gram_schmidt()` and manual normalization
4. QR factorization in `AA` and verification `A = QR`
5. Orthogonal matrix verification — rotation by π/6 symbolically
6. Least squares — exact normal equations on rational data
7. Polynomial fitting via Vandermonde + normal equations
8. The L² inner product on `PolynomialRing(QQ)` — Legendre polynomials by Gram–Schmidt
9. Project `x³` onto degree-≤ 2 polynomials — verify the `(3/5) x` answer from worked example E10
10. **Exercises** with solutions


## 1. Coordinates in an ONB by dot products

Worked example E1: ONB `q₁ = (2, 2, 1)/3, q₂ = (2, −1, −2)/3, q₃ = (1, −2, 2)/3`. Coordinates of `v = (5, 0, 4)`.

In [ ]:
q1 = vector(QQ, [2, 2, 1]) / 3
q2 = vector(QQ, [2, -1, -2]) / 3
q3 = vector(QQ, [1, -2, 2]) / 3
Q = column_matrix([q1, q2, q3])

show('QᵀQ ='); show(Q.transpose() * Q)
print('All entries are exactly 0 or 1 — Q is orthonormal.')

v = vector(QQ, [5, 0, 4])
coords = vector(QQ, [v.dot_product(qi) for qi in [q1, q2, q3]])
show('[v]_𝓠 =', coords)
show('reconstructed:', sum(c * qi for c, qi in zip(coords, [q1, q2, q3])))

## 2. Projection onto a line and onto a plane

(a) Worked example E2: project `b = (3, 5, 7)` onto the line through 0 in direction `u = (1, 2, 2)`.

In [ ]:
b = vector(QQ, [3, 5, 7])
u = vector(QQ, [1, 2, 2])

proj = (b.dot_product(u) / u.dot_product(u)) * u
show('proj_u(b) =', proj)
show('residual b - proj =', b - proj)
show('residual · u =', (b - proj).dot_product(u), '   (must be 0)')

(b) Worked example E3: project `b = (1, 0, 0)` onto `V = span((1, 1, 0), (0, 1, 1))`. Use the projection-matrix formula `P = A(AᵀA)⁻¹Aᵀ`.

In [ ]:
A = column_matrix([vector(QQ, [1, 1, 0]), vector(QQ, [0, 1, 1])])
b = vector(QQ, [1, 0, 0])

P = A * (A.transpose() * A).inverse() * A.transpose()
show('Projection matrix P ='); show(P)

proj_V_b = P * b
show('proj_V(b) =', proj_V_b, '   (expected (2/3, 1/3, -1/3))')
show('residual =', b - proj_V_b, '   (expected (1/3, -1/3, 1/3))')

# P should be symmetric and idempotent
show('P symmetric?', P == P.transpose())
show('P idempotent (P² = P)?', P * P == P)

## 3. Gram–Schmidt symbolically

Worked example E5 vectors `v₁ = (1, 1, 0, 0), v₂ = (1, 0, 1, 0), v₃ = (0, 0, 1, 1)`.

Sage's `M.gram_schmidt()` operates on **rows** and returns `(G, mu)` where `G` has *orthogonal* (not orthonormal) rows. We then normalize by hand.

In [ ]:
V_rows = matrix(QQ, [
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 0, 1, 1],
])
G, mu = V_rows.gram_schmidt()
show('Orthogonal rows G ='); show(G)
show('mu (lower-triangular Gram-Schmidt coefficients) ='); show(mu)

In [ ]:
# Normalize each row by its length to get an ONB. Norms involve sqrt -> AA.
ortho_rows = list(G)
norms = [vec.norm() for vec in ortho_rows]
show('norms (in AA) =', norms)

# Build the ONB matrix in AA
Q_AA = matrix(AA, [vec / vec.norm() for vec in ortho_rows])
show('Q (orthonormal rows, in AA) ='); show(Q_AA)
show('Q · Qᵀ ='); show(Q_AA * Q_AA.transpose())

## 4. QR factorization in `AA`

Use Sage's built-in `.QR()` on the `AA` matrix to produce `A = QR` exactly (with algebraic-real entries). Cross-check the product.

In [ ]:
# A as columns -> we transpose Q_AA (which had orthonormal rows) into a column matrix.
A_AA = matrix(AA, [
    [1, 1, 0],
    [1, 0, 0],
    [0, 1, 1],
    [0, 0, 1],
])
show('A ='); show(A_AA)

Q_qr, R_qr = A_AA.QR()
show('Q (orthonormal columns) ='); show(Q_qr)
show('R (upper triangular) ='); show(R_qr)

show('Q · R ='); show(Q_qr * R_qr)
show('Match A?', (Q_qr * R_qr) == A_AA)

## 5. Orthogonal matrix — rotation by π/6 symbolically

In [ ]:
theta = pi / 6
R = matrix(SR, [
    [cos(theta), -sin(theta)],
    [sin(theta),  cos(theta)],
])
show('R(π/6) ='); show(R)
show('Rᵀ R ='); show((R.transpose() * R).simplify_full())
show('det(R) =', R.det().simplify_full())

# Length-preservation on v = (3, 4)
v = vector(SR, [3, 4])
Rv = R * v
show('||v||² =', v.dot_product(v))
show('||Rv||² =', (Rv.dot_product(Rv)).simplify_full())

## 6. Least squares — exact normal equations

Worked example E8: fit `y = a + bx` to `(0, 1), (1, 3), (2, 4), (3, 4)`.

In [ ]:
xs = vector(QQ, [0, 1, 2, 3])
ys = vector(QQ, [1, 3, 4, 4])

A = column_matrix([vector(QQ, [1] * len(xs)), xs])
show('A ='); show(A)
show('AᵀA ='); show(A.transpose() * A)
show('Aᵀy ='); show(A.transpose() * ys)

xhat = (A.transpose() * A).solve_right(A.transpose() * ys)
show('Least-squares (a, b) =', xhat, '   (expected (9/5, 4/5))')

In [ ]:
# Residuals and their sum (the latter is forced to 0 by Aᵀ having a row of 1s)
residuals = ys - A * xhat
show('Residuals =', residuals)
show('Σ residuals =', sum(residuals))

## 7. Polynomial fitting — Vandermonde matrix

Worked example E9: fit `y = a + bx + cx²` to `(−1, 1), (0, 0), (1, 0), (2, −2)`.

In [ ]:
xs = vector(QQ, [-1, 0, 1, 2])
ys = vector(QQ, [1, 0, 0, -2])

# Vandermonde: columns 1, x, x^2
A = column_matrix([
    vector(QQ, [1, 1, 1, 1]),
    xs,
    vector(QQ, [x^2 for x in xs]),
])
show('Vandermonde A ='); show(A)

xhat = (A.transpose() * A).solve_right(A.transpose() * ys)
show('Least-squares (a, b, c) =', xhat)

## 8. The L² inner product on `PolynomialRing(QQ)` — Legendre polynomials

`⟨p, q⟩ := ∫_{-1}^{1} p(x) q(x) dx`. We compute this exactly in Sage by computing the antiderivative and evaluating at ±1.

In [ ]:
R.<x> = PolynomialRing(QQ)

def L2_inner_product(p, q, lo=-1, hi=1):
    """Exact L² inner product of polynomials p and q over [lo, hi]."""
    F = (p * q).integral(x)
    return F.subs(x=hi) - F.subs(x=lo)

# Sanity: ⟨1, 1⟩ on [-1, 1] should be 2. ⟨x, x⟩ should be 2/3.
show('⟨1, 1⟩ =', L2_inner_product(R(1), R(1)))
show('⟨x, x⟩ =', L2_inner_product(x, x))
show('⟨1, x⟩ =', L2_inner_product(R(1), x), '   (orthogonal — odd function on symmetric interval)')

In [ ]:
def gs_L2_orthogonal(monomials):
    """Gram-Schmidt under L² — return ORTHOGONAL (not normalized) polynomials.
    Stays inside PolynomialRing(QQ) so the answers are clean rationals."""
    out = []
    for v in monomials:
        u = v
        for q in out:
            u = u - (L2_inner_product(u, q) / L2_inner_product(q, q)) * q
        out.append(u)
    return out

# Run on monomials 1, x, x^2, x^3
monos = [R(1), x, x^2, x^3]
P_ortho = gs_L2_orthogonal(monos)
print('Orthogonal Legendre polynomials L_0, …, L_3 (unnormalized):')
for k, Lk in enumerate(P_ortho):
    show(f'L_{k}(x) =', Lk)

In [ ]:
# Cross-check: pairwise orthogonality ⟨L_i, L_j⟩ = 0 for i ≠ j; diagonal entries are ⟨L_k, L_k⟩.
print('Gram matrix (diagonal — orthogonal!):')
G_check = matrix(QQ, [[L2_inner_product(pi, pj) for pj in P_ortho] for pi in P_ortho])
show(G_check)
print()
print('To match the textbook normalization with leading coefficient = 1, we get exactly')
print('the Legendre polynomials of the second kind: L_2 = x^2 - 1/3, L_3 = x^3 - (3/5)x.')
print('To get the standard Legendre P_n (with P_n(1) = 1), multiply each L_n by a rational scalar.')

## 9. Projecting `x³` onto degree-≤ 2 polynomials

Worked example E10: the L² projection of `x³` onto `span(1, x, x²)` is `(3/5) x`.

In [ ]:
f = x^3
P3 = P_ortho[:3]  # L_0, L_1, L_2 — orthogonal basis of degree-≤ 2 polynomials

# Projection formula for ORTHOGONAL (not orthonormal) basis:
#   proj_V(f) = Σ ⟨f, L_k⟩ / ⟨L_k, L_k⟩ · L_k
proj = sum((L2_inner_product(f, Lk) / L2_inner_product(Lk, Lk)) * Lk for Lk in P3)
show('proj_{P_2}(x³) =', proj, '   (expected (3/5) x)')

In [ ]:
# Residual x^3 - proj should be orthogonal to 1, x, x^2
residual = f - proj
print('Verify residual ⊥ 1, x, x²:')
for w in [R(1), x, x^2]:
    show(f'⟨x³ - proj, {w}⟩ =', L2_inner_product(residual, w))

---

## 10. Exercises (with solutions)

Each exercise lets you cross-check a hand calculation symbolically.

### E1 — Verify an ONB and compute coordinates

Show that `q₁ = (1, 2, 2, 0)/3, q₂ = (2, −2, 1, 0)/3, q₃ = (0, 0, 0, 1)` is an orthonormal set in ℝ⁴. Find the coordinates of `v = (5, −2, 4, 4)` (which happens to lie in `V = span(q₁, q₂, q₃)`) in this basis.

### E2 — Project onto a 2D plane in ℝ⁴

Let `V = span((1, 1, 0, 0), (1, 0, 1, 0))` ⊂ ℝ⁴. Find the projection matrix `P = A(AᵀA)⁻¹Aᵀ` and the projection of `b = (1, 1, 1, 1)`.

### E3 — Gram–Schmidt on `(1, 1, 1), (1, 1, 0), (1, 0, 0)`

Apply Gram–Schmidt to these three vectors in ℝ³. (The output is *not* the standard basis — these vectors don't have the upper-triangular structure of Exercise 5 from `exercises.md`.)

### E4 — QR factorization

Compute the QR factorization of `A = [[1, 2], [1, 0], [1, 1]]` exactly (using `AA`).

### E5 — Legendre polynomial of degree 4

Extend the (unnormalized) Gram–Schmidt computation in §8 to include `x⁴`. The result `L₄` should be a *rational* scalar multiple of the standard Legendre polynomial `P₄(x) = (35x⁴ − 30x² + 3) / 8`. (Specifically, `L₄ = (8/35) P₄`.)

---

### Solutions

### A1

In [ ]:
q1 = vector(QQ, [1, 2, 2, 0]) / 3
q2 = vector(QQ, [2, -2, 1, 0]) / 3
q3 = vector(QQ, [0, 0, 0, 1])

show('q1 · q1 =', q1.dot_product(q1), '   ‖q1‖² should be 1')
show('q2 · q2 =', q2.dot_product(q2))
show('q3 · q3 =', q3.dot_product(q3))
show('q1 · q2 =', q1.dot_product(q2))
show('q1 · q3 =', q1.dot_product(q3))
show('q2 · q3 =', q2.dot_product(q3))

v = vector(QQ, [5, -2, 4, 4])
coords = vector(QQ, [v.dot_product(qi) for qi in [q1, q2, q3]])
show('[v]_𝓠 =', coords, '   (expected (3, 6, 4))')
show('reconstructed:', sum(c * qi for c, qi in zip(coords, [q1, q2, q3])),
     '   (matches v ⇒ v ∈ V)')

### A2

In [ ]:
A = column_matrix([vector(QQ, [1, 1, 0, 0]), vector(QQ, [1, 0, 1, 0])])
b = vector(QQ, [1, 1, 1, 1])

P = A * (A.transpose() * A).inverse() * A.transpose()
show('Projection matrix P ='); show(P)
show('proj_V(b) =', P * b)
show('residual =', b - P * b)
# Sanity: residual should be in V^perp -> dot with each column of A is 0
show('residual · column 1 =', (b - P*b).dot_product(A.column(0)))
show('residual · column 2 =', (b - P*b).dot_product(A.column(1)))

### A3

In [ ]:
V_rows = matrix(QQ, [[1, 1, 1], [1, 1, 0], [1, 0, 0]])
G, mu = V_rows.gram_schmidt()
show('Orthogonal rows G ='); show(G)

# Normalize to get ONB
Q_AA = matrix(AA, [vec / vec.norm() for vec in G])
show('Orthonormal rows (in AA) ='); show(Q_AA)
show('Q · Qᵀ ='); show(Q_AA * Q_AA.transpose())

### A4

In [ ]:
A = matrix(AA, [[1, 2], [1, 0], [1, 1]])
Q_qr, R_qr = A.QR()
show('Q ='); show(Q_qr)
show('R ='); show(R_qr)
show('Q · R ='); show(Q_qr * R_qr)
show('Match A?', (Q_qr * R_qr) == A)

### A5

In [ ]:
# Re-run unnormalized Gram-Schmidt on monomials up to x^4
monos = [R(1), x, x^2, x^3, x^4]
P_ortho_5 = gs_L2_orthogonal(monos)
show('L_4(x) (Sage) =', P_ortho_5[4])

# Standard Legendre polynomial P_4(x) = (35 x^4 - 30 x^2 + 3) / 8
# Predicted L_4 = (8/35) * P_4 = x^4 - (6/7) x^2 + 3/35
predicted_L4 = (8/35) * (35*x^4 - 30*x^2 + 3) / 8
show('L_4(x) (predicted) =', predicted_L4.expand())
show('Match?', P_ortho_5[4] == predicted_L4)

---

## Where to next

- **Chapter 8 (Sage):** **Determinants** — the signed-volume scalar that characterizes invertibility from one number, sets up Cramer's rule, and produces the **characteristic polynomial** for eigenvalues in Ch 9. Sage's `Matrix.determinant()` works over `QQ`, `SR`, `PolynomialRing`, and more.
- **Chapter 9 (Sage):** **Eigenvalues and diagonalization** — the Ch 7 ONB machinery shines for *symmetric* matrices, where the **spectral theorem** gives an orthonormal basis of eigenvectors. Sage's `Matrix.eigenvectors_right()` and `.diagonalization()` give exact answers when eigenvalues lie in algebraic extensions.
